In [4]:
import pandas as pd
from IPython.display import display

# Define the data
data = {
    "Model": ["Transfer learnt model built on a pretrained LLM"] * 7,
    "Tasks and Comments": [
        "Preprocessing Steps - 1 Cleaning, 2.Tokenization,3.Normalize",
        "Training - Model built, train and test handled correctly?",
        "Evaluation - ROUGE-L Score, BERT Score?",
        "Interpretation using Lime",
        "1st round of tuning - What was the issue faced/tuned?",
        "2nd round of tuning - What was the issue faced/tuned?",
        "Final AUC value? Next steps Recommended?"
    ],
    "Status": ["Done","Done","Pending","Pending","Pending","Pending","Pending"] ,
    "Individual Responsible": ["Anjitha", "Anjitha", "", "", "", "", ""]
}

# Create DataFrame
df = pd.DataFrame(data)

# Display the table
display(df)


,Model,Tasks and Comments,Status,Individual Responsible
0,Transfer learnt model built on a pretrained LLM,"Preprocessing Steps - 1 Cleaning, 2.Tokenizati...",Done,Anjitha
1,Transfer learnt model built on a pretrained LLM,"Training - Model built, train and test handled...",Done,Anjitha
2,Transfer learnt model built on a pretrained LLM,"Evaluation - ROUGE-L Score, BERT Score?",Pending,
3,Transfer learnt model built on a pretrained LLM,Interpretation using Lime,Pending,
4,Transfer learnt model built on a pretrained LLM,1st round of tuning - What was the issue faced...,Pending,
5,Transfer learnt model built on a pretrained LLM,2nd round of tuning - What was the issue faced...,Pending,
6,Transfer learnt model built on a pretrained LLM,Final AUC value? Next steps Recommended?,Pending,


In [1]:
# Install necessary libraries
!pip install transformers datasets peft accelerate rouge-score bert-score lime datasets evaluate torch bitsandbytes datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 22.7 MB/s eta 0:00:00:00:0100:01
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=4a124cfb8c39a73c0afb39dd3ab14a84e59a30303595b2fb8940befd6331e9fe
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [2]:
!pip install evaluate


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import random
import re
import pandas as pd

In [4]:
# Function to load CSV with error handling
def load_csv_with_error_handling(file_path):
    try:
        df = pd.read_csv(file_path, on_bad_lines='skip', encoding="utf-8")
        print(f"{file_path} loaded successfully!")
        return df
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load the train and valid CSV files
df_train = load_csv_with_error_handling("/kaggle/input/dataset/train.csv")
df_valid = load_csv_with_error_handling("/kaggle/input/dataset/valid.csv")

/kaggle/input/dataset/train.csv loaded successfully!
/kaggle/input/dataset/valid.csv loaded successfully!


In [5]:
# If loading is successful, save cleaned versions
if df_train is not None and df_valid is not None:
    # Save cleaned versions of the train and valid datasets
    train_fixed_path = "/kaggle/working/train_fixed.csv"
    valid_fixed_path = "/kaggle/working/valid_fixed.csv"

    df_train.to_csv(train_fixed_path, index=False)
    df_valid.to_csv(valid_fixed_path, index=False)

    print("Cleaned files saved as train_fixed.csv and valid_fixed.csv")

    # Load the cleaned datasets using Hugging Face's datasets library
    dataset = load_dataset("csv", data_files={"train": train_fixed_path, "validation": valid_fixed_path})

    # Take a random 10% sample of the dataset
    def sample_dataset(dataset, sample_percentage=0.1):
        train_sample = dataset['train'].shuffle(seed=42).select([i for i in range(int(len(dataset['train']) * sample_percentage))])
        valid_sample = dataset['validation'].shuffle(seed=42).select([i for i in range(int(len(dataset['validation']) * sample_percentage))])
        return train_sample, valid_sample

    train_sample, valid_sample = sample_dataset(dataset)
    
    print("Sampled 10% of the dataset for training and validation.")

else:
    print("Error: One or both of the CSV files could not be loaded. Exiting.")

# Load tokenizer and model (use distilgpt2 for faster training)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

# Normalize Text function
def normalize_text(text):
    if text is None:
        return ""  # Replace None values with an empty string
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

# Preprocess dataset
train_sample = train_sample.map(lambda x: {
    "query": normalize_text(x.get("query")),
    "finalpassage": normalize_text(x.get("finalpassage"))
})

valid_sample = valid_sample.map(lambda x: {
    "query": normalize_text(x.get("query")),
    "finalpassage": normalize_text(x.get("finalpassage"))
})

# Tokenization function
def preprocess_function(examples):
    inputs = tokenizer(examples["query"], padding="max_length", truncation=True, max_length=128)
    targets = tokenizer(examples["finalpassage"], padding="max_length", truncation=True, max_length=128)

    inputs["labels"] = targets["input_ids"]  # Set targets' input_ids as labels
    return inputs

# Apply preprocessing to the dataset
train_sample_tokenized = train_sample.map(preprocess_function, batched=True)
valid_sample_tokenized = valid_sample.map(preprocess_function, batched=True)


Cleaned files saved as train_fixed.csv and valid_fixed.csv


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Sampled 10% of the dataset for training and validation.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1058 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1058 [00:00<?, ? examples/s]

In [6]:
# Load the distilgpt2 model
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# PEFT (Low-Rank Adaptation) configuration for efficient fine-tuning
peft_config = LoraConfig(
    r=8, lora_alpha=32, lora_dropout=0.1, bias="none"
)

model = get_peft_model(model, peft_config)  # Wrap the model with PEFT

# Define device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/peft/tuners/lora/layer.py:1264: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


PeftModel(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-5): 6 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2SdpaAttention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lor

In [ ]:
!nvidia-smi


In [7]:
import torch
from transformers import Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

torch.cuda.empty_cache()


# Training arguments
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=500,
    fp16=True,  # Enable mixed precision training for GPU
    report_to="none",  # Disable unnecessary logging systems
)

print("Training arguments set successfully!")

# Trainer setup
print("Setting up the trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_sample_tokenized,
    eval_dataset=valid_sample_tokenized,
    tokenizer=tokenizer,
)

print("Trainer set up successfully!")

# Start the training process
print("Starting the training process...")

# Add progress printing
trainer.train()

# Add checkpoint prints during training
def print_train_loss(trainer):
    for epoch in range(training_args.num_train_epochs):
        print(f"Starting epoch {epoch + 1}...")
        trainer.train()
        print(f"Completed epoch {epoch + 1}")
        print(f"Training loss at the end of epoch {epoch + 1}: {trainer.state.global_step}")

# Monitor progress
print("Training completed!")
torch.cuda.empty_cache()

Using device: cuda
Setting up training arguments...
Training arguments set successfully!
Setting up the trainer...
Trainer set up successfully!
Starting the training process...


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-7-50c346dfbce0>:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,7.039800,No log
2,6.975400,No log
3,6.804200,No log


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

Training completed!


In [ ]:
from transformers import Trainer, TrainingArguments
import evaluate
import torch
import os
torch.cuda.empty_cache()
# Ensure we're using the CPU
device = torch.device("cuda")
model.to(device)
print(f"Using device: {device}")


# Load ROUGE and BERTScore metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Evaluation function for CPU
def evaluate_model(model, tokenizer, eval_dataset):
    print("Evaluating the model...")
    print(f"Using device: {device}")
    # Ensure the model is on CPU
    model.to(device)

    # Create training arguments with reduced batch size
    training_args = TrainingArguments(
    output_dir='./results',
    do_train=False,  # No training, just evaluation
    do_eval=True,
    per_device_eval_batch_size=1,  # Use smaller batch size if needed
    no_cuda=True  # Force the trainer to avoid using CUDA
)

    
    # Create Trainer with the training arguments
    trainer = Trainer(
        model=model,
        args=training_args,  # Pass the training arguments
        tokenizer=tokenizer,
        eval_dataset=eval_dataset  # Ensure eval_dataset is passed here
    )

    # Generate predictions and labels for evaluation
    predictions, references = trainer.predict(eval_dataset)

    # Convert the predictions to strings if necessary (assuming predictions are token IDs)
    predicted_texts = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    reference_texts = tokenizer.batch_decode(references, skip_special_tokens=True)
    
    # Evaluate ROUGE
    rouge_results = rouge.compute(predictions=predicted_texts, references=reference_texts)
    print(f"ROUGE Results: {rouge_results}")
    
    # Evaluate BERTScore
    bertscore_results = bertscore.compute(predictions=predicted_texts, references=reference_texts)
    print(f"BERTScore Results: {bertscore_results}")

# Save the trained model
def save_model(model, tokenizer, save_directory="./saved_model"):
    print("Saving the model...")
    model.save_pretrained(save_directory)
    tokenizer.save_pretrained(save_directory)
    print(f"Model and tokenizer saved to {save_directory}")

# Save the model
save_model(model, tokenizer)
torch.cuda.empty_cache()
# Now evaluate the model (replace 'valid_sample_tokenized' with your validation dataset)
evaluate_model(model, tokenizer, valid_sample_tokenized)


Using device: cuda


Saving the model...
Model and tokenizer saved to ./saved_model
Evaluating the model...
Using device: cuda


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1590: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
<ipython-input-8-86d717682cd9>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
